# AlexNet on Tiny-ImageNet — FP32, Architecture Variants & QAT (INT8)

**Author:** Rafael
**Dataset:** Tiny-ImageNet-200 (200 classes, 64×64 RGB)
**Goal:** Train and compare three AlexNet variants in FP32, then apply
Quantization-Aware Training (QAT) to obtain INT8 versions. Report **Top-1**
and **Top-5** accuracy for every model.

## Notebook layout

1. Configuration & reproducibility
2. Dataset & loaders
3. Model definitions (`AlexNet`, `AlexNet3x3`, `AlexNetSmallKernel`)
4. QAT helpers (fuse + prepare)
5. FP32 training (3 models)
6. QAT training (3 models) and INT8 conversion
7. Evaluation — Top-1 and Top-5 — for FP32 and INT8
8. Final comparison table & ranking


## 1. Configuration & reproducibility

Everything that controls the experiment lives in a single `ExperimentConfig`
dict. The seed is fixed and `set_seed` propagates it to `random`, `numpy`,
and `torch` (CPU & CUDA).


In [1]:
# Standard library
import os
import random
import sys
from dataclasses import asdict, replace
from pathlib import Path

# Third-party
import numpy as np
import torch
import torch.nn as nn
import torchinfo

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Local — ml package
from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    MODEL_REGISTRY, register_model,
    Trainer, make_qat_callback,
    build_qat, load_best_model, convert_to_int8,
    build_comparison_table, create_results_summary, disk_mb,
)
from configs.loader import load_config

# Model architectures
from models import AlexNetTV, AlexNet3x3, AlexNetSmallKernel, build_alexnet

torch.backends.quantized.engine = "fbgemm"

/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR = project_root / "checkpoints" / "alexnet_qat"
RESULTS_DIR = project_root / "results" / "alexnet_qat"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ─── Hyperparameters — loaded from configs/, override here if needed ──────────
data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED  # tie dataset split seed to notebook-level SEED constant
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
qat_cfg = QATConfig(**load_config("qat.yaml"))
# ─────────────────────────────────────────────────────────────────────────────
print(device)

cuda


## 2. Dataset & loaders

We use Tiny-ImageNet-200 from Kaggle. The train folder is split 90/10 into
train/val with a **seeded generator** so the split is identical on every run.

* Training transforms: light augmentation (random crop + hflip + color jitter)
* Validation transforms: deterministic resize + center crop

Note that we build two `ImageFolder` objects (one per transform) and index
them with the same `Subset` indices — this keeps train-time augmentation
separate from val-time deterministic preprocessing.


In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path
print("Tiny-ImageNet train path:", train_path)

Tiny-ImageNet train path: /home/rafael/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1/tiny-imagenet-200/train


In [4]:
train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {data_cfg.num_classes}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

Train samples: 90,000
Val   samples: 10,000
Classes:       200
Batches/epoch: train=1406, val=157


## 3. Model architectures

Three variants of AlexNet are compared:

| Name | Backbone | Notes |
|---|---|---|
| `AlexNet_FP32` | torchvision `alexnet(IMAGENET1K_V1)` | Last FC replaced by `Linear(4096, 200)`. Pretrained, fine-tuned. |
| `AlexNet3x3` | All 3×3 conv stack, AdaptiveAvgPool(6×6) → 3-layer MLP | From-scratch, AlexNet-style head. |
| `AlexNetSmallKernel` | All 3×3 convs, AdaptiveAvgPool(1×1) → single `Linear(256, 200)` | Lightweight, ~250× smaller classifier. |


In [5]:
# Model classes are defined in models/; run a quick shape + param check
x = torch.randn(2, 3, data_cfg.img_size, data_cfg.img_size)
for ctor in (build_alexnet, AlexNet3x3, AlexNetSmallKernel):
    m = ctor().eval()
    with torch.no_grad():
        y = m(x)
    assert y.shape == (2, data_cfg.num_classes)
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    print(f"{ctor.__name__:25s} OK -> {tuple(y.shape)}, params={info.total_params/1e6:.2f}M")

build_alexnet             OK -> (2, 200), params=57.82M
AlexNet3x3                OK -> (2, 200), params=57.61M
AlexNetSmallKernel        OK -> (2, 200), params=1.60M


## 5. QAT helpers 

In the original notebook the QAT prep was duplicated three times. Here it
is one function that:

1. puts the model in `train()` mode (required by `prepare_qat`),
2. sets the fbgemm `qconfig`,
3. fuses a user-supplied list of `[conv, relu]` patterns,
4. returns `prepare_qat(model)`.

Only Conv+ReLU (or Conv+BN+ReLU) is fused — never anything involving
`MaxPool` or `AdaptiveAvgPool`, which is invalid and crashes `fuse_modules`.


In [6]:
# Fuse maps for each architecture — Conv+ReLU pairs by index inside .features
FUSE_MAP_ALEXNET_TV = [["0","1"],["3","4"],["6","7"],["8","9"],["10","11"]]
FUSE_MAP_3X3        = [["0","1"],["3","4"],["6","7"],["8","9"],["10","11"]]
FUSE_MAP_SMALL      = [["0","1"],["3","4"],["6","7"],["8","9"],["10","11"]]

MODEL_REGISTRY.clear()
register_model("alexnet_fp32",        build_alexnet,        fuse_map=FUSE_MAP_ALEXNET_TV, fuse_root_attr="features")
register_model("alexnet_3x3",         AlexNet3x3,           fuse_map=FUSE_MAP_3X3,        fuse_root_attr="features")
register_model("alexnet_small_kernel",AlexNetSmallKernel,   fuse_map=FUSE_MAP_SMALL,      fuse_root_attr="features")
print("Registered:", list(MODEL_REGISTRY))

Registered: ['alexnet_fp32', 'alexnet_3x3', 'alexnet_small_kernel']


## 7. FP32 training — three architectures

In [7]:
fp32_models = {}
fp32_training_results = {}

for name, spec in MODEL_REGISTRY.items():
    model_cfg = replace(fp32_cfg, lr=spec["lr"]) if "lr" in spec else fp32_cfg
    print("=" * 72)
    print(f"Training: {name}  lr={model_cfg.lr}  epochs={model_cfg.epochs}")
    print("=" * 72)

    model = spec["ctor"]().to(device)

    trainer = Trainer(
        model, train_loader, val_loader, model_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    results = trainer.fit()

    fp32_training_results[name] = results
    fp32_models[name] = model.cpu()

    del model
    torch.cuda.empty_cache()

print("\nFP32 training complete.")

Training: alexnet_fp32  lr=0.0003  epochs=10


Validation: 100%|██████████| 157/157 [00:03<00:00, 48.30it/s, acc=13.93%, loss=4.2207]


Epoch   1/10 | train_loss=4.6628 train_acc=7.48% | val_loss=4.2207 val_acc=13.93% | time=52.1s


Training:  16%|█▌        | 220/1406 [00:08<00:43, 27.42it/s, acc=10.38%, loss=4.4317]


KeyboardInterrupt: 

## 8. Quantization-Aware Training (QAT)

We fine-tune each FP32 checkpoint for a few epochs with fake-quant observers
inserted, then `convert` to true INT8 on CPU.

> **Note on the engine.** We set `torch.backends.quantized.engine = "fbgemm"`
> at the top of the notebook. QAT *training* happens on GPU; `convert` and
> INT8 *inference* must run on CPU.


In [ ]:
qat_train_cfg = replace(
    fp32_cfg,
    epochs=qat_cfg.epochs,
    lr=qat_cfg.lr,
    weight_decay=qat_cfg.weight_decay,
    use_amp=False,  # AMP is incompatible with QAT fake-quant observers
)

qat_models = {}
qat_training_results = {}

for name in MODEL_REGISTRY:
    model_lr = spec.get("lr", fp32_cfg.lr)
    print("=" * 72)
    print(f"QAT fine-tuning: {name}")
    print("=" * 72)

    qat_model = build_qat(name, save_dir=SAVE_DIR, device=device)
    cb = make_qat_callback(qat_cfg.freeze_bn_epoch, qat_cfg.disable_observer_epoch)

    trainer = Trainer(
        qat_model, train_loader, val_loader, qat_train_cfg,
        device, SAVE_DIR, f"qat_{name}", num_classes=data_cfg.num_classes,
        epoch_callback=cb,
    )
    results = trainer.fit()

    qat_training_results[name] = results
    qat_models[name] = qat_model.cpu()

    del qat_model
    torch.cuda.empty_cache()

print("\nQAT training complete.")

QAT fine-tuning: alexnet_fp32


/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(
Validation: 100%|██████████| 157/157 [00:01<00:00, 126.05it/s]


Epoch   1/2 | Train 3.8471/22.28% | Val 3.5085/29.82% | 50.0s


Validation: 100%|██████████| 157/157 [00:01<00:00, 126.16it/s]


Epoch   2/2 | Train 3.7980/23.37% | Val 3.4886/30.26% | 50.0s
QAT fine-tuning: alexnet_3x3


...

Epoch   2/2 | Train 5.2983/0.52% | Val 5.2989/0.31% | 56.8s
QAT fine-tuning: alexnet_small_kernel


Validation: 100%|██████████| 157/157 [00:01<00:00, 90.30it/s]


Epoch   1/2 | Train 4.6313/7.86% | Val 4.5148/9.20% | 42.9s


Validation: 100%|██████████| 157/157 [00:01<00:00, 90.06it/s]

Epoch   2/2 | Train 4.6106/8.20% | Val 4.5112/9.38% | 42.6s

QAT training complete.


## 9. INT8 conversion and CPU evaluation

`tq.convert` materialises true quantised ops; this must run on CPU and the
model must be in `eval()` mode. Because INT8 inference is CPU-only, we
build a small CPU-side val loader (`num_workers=0` is safer here — fewer
chances of hangs after a GPU run) and reuse `evaluate_topk`.


In [ ]:
val_loader_cpu = torch.utils.data.DataLoader(
    val_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False,
)
cpu_device = torch.device("cpu")

int8_models = {}
for name in MODEL_REGISTRY:
    int8_models[name] = convert_to_int8(qat_models[name].cpu().eval())

for name, m in int8_models.items():
    torch.save(m.state_dict(), SAVE_DIR / f"{name}.pth")
print("INT8 conversion done.")

int8_metrics = {}
for name, m in int8_models.items():
    m = m.cpu().eval()
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(m, val_loader_cpu, val_loader_cpu, dummy_cfg, cpu_device, SAVE_DIR, name,
                      num_classes=data_cfg.num_classes)
    metrics = trainer.evaluate(topk=(1, 5))
    int8_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

INT8 conversion done.
alexnet_fp32           | loss=3.1124 | top1=30.16% | top5=56.77%
alexnet_3x3            | loss=5.2989 | top1=0.50% | top5=2.50%
alexnet_small_kernel   | loss=4.3141 | top1=9.53% | top5=27.48%


## 10. FP32 evaluation — reload best checkpoints

Reload each best FP32 checkpoint from disk to make sure we're evaluating
the *best* epoch (not whatever was in memory at the end of training).


In [ ]:
fp32_models = {
    "AlexNet_FP32":      load_best_model("alexnet_fp32",         build_alexnet,        SAVE_DIR, device),
    "AlexNet3x3_FP32":   load_best_model("alexnet_3x3",          AlexNet3x3,           SAVE_DIR, device),
    "AlexNetSmall_FP32": load_best_model("alexnet_small_kernel", AlexNetSmallKernel,   SAVE_DIR, device),
}

fp32_metrics = {}
for name, m in fp32_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(m, train_loader, val_loader, dummy_cfg, device, SAVE_DIR, name,
                      num_classes=data_cfg.num_classes)
    metrics = trainer.evaluate(topk=(1, 5))
    fp32_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

AlexNet_FP32           | loss=3.2539 | top1=27.16% | top5=53.51%
AlexNet3x3_FP32        | loss=5.2989 | top1=0.50% | top5=2.50%
AlexNetSmall_FP32      | loss=4.3664 | top1=8.68% | top5=25.36%


## 11. Final comparison — Top-1, Top-5, size, params

This is the single artefact to read at the end:

* **Top-1 / Top-5 accuracy** on the validation split
* **Trainable parameters** (millions)
* **Disk size** of the checkpoint (MB) — for INT8 this is the *real* on-disk size


In [ ]:
rows = []

fp32_files = {
    "AlexNet_FP32":      "alexnet_fp32_best.pth",
    "AlexNet3x3_FP32":   "alexnet_3x3_best.pth",
    "AlexNetSmall_FP32": "alexnet_small_kernel_best.pth",
}
for name, m in fp32_models.items():
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": name, "precision": "FP32",
        "top1_%": fp32_metrics[name]["top1"], "top5_%": fp32_metrics[name]["top5"],
        "loss": fp32_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB": disk_mb(SAVE_DIR / fp32_files[name]),
    })

int8_name_map = {
    "alexnet_fp32": "AlexNet_FP32",
    "alexnet_3x3":  "AlexNet3x3_FP32",
    "alexnet_small_kernel": "AlexNetSmall_FP32",
}
for name, m in int8_models.items():
    fp32_info = torchinfo.summary(fp32_models[int8_name_map[name]],
                                   input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": f"{name}_INT8", "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"], "top5_%": int8_metrics[name]["top5"],
        "loss": int8_metrics[name]["loss"],
        "params_M": fp32_info.total_params / 1e6,
        "size_MB": disk_mb(SAVE_DIR / f"{name}.pth"),
    })

df = build_comparison_table(rows)
df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)
df

OSError: Cannot save file into a non-existent directory: '/home/rafael/Documents/alexnet_rafael/results/alexnet_qat'

In [ ]:
print("=" * 72)
print("RANKING BY TOP-1 ACCURACY (all precisions)")
print("=" * 72)
ranked = df.sort_values("top1_%", ascending=False).reset_index(drop=True)
for i, row in ranked.iterrows():
    print(f"{i+1:2d}. {row['model']:22s} [{row['precision']}] | "
          f"top1={row['top1_%']:6.2f}% | top5={row['top5_%']:6.2f}% | "
          f"params={row['params_M']:6.2f}M | size={row['size_MB']:6.2f}MB")


RANKING BY TOP-1 ACCURACY (all precisions)
 1. alexnet_fp32_INT8      [INT8] | top1= 30.22% | top5= 56.78% | params= 57.82M | size= 55.33MB
 2. AlexNet_FP32           [FP32] | top1= 27.30% | top5= 53.65% | params= 57.82M | size=661.75MB
 3. alexnet_small_kernel_INT8 [INT8] | top1=  9.53% | top5= 27.48% | params=  1.60M | size=  1.56MB
 4. AlexNetSmall_FP32      [FP32] | top1=  8.68% | top5= 25.36% | params=  1.60M | size= 18.35MB
 5. alexnet_3x3_INT8       [INT8] | top1=  7.93% | top5= 23.48% | params= 57.61M | size= 55.12MB
 6. AlexNet3x3_FP32        [FP32] | top1=  6.82% | top5= 21.20% | params= 57.61M | size=659.26MB


## 12. Persist experiment summary

A single JSON next to the checkpoints captures: config, system info,
per-model metrics. Re-running this notebook with the same seed should
reproduce these numbers up to non-determinism in cuDNN kernels (we set
`benchmark=True` for speed; flip to `False` for exact bit reproducibility).


In [ ]:
create_results_summary(
    results={
        "fp32_metrics": fp32_metrics,
        "int8_metrics": int8_metrics,
        "fp32_training_results": fp32_training_results,
        "qat_training_results": qat_training_results,
    },
    config=asdict(fp32_cfg),
    output_path=RESULTS_DIR / "experiment_summary.json",
)